# 06 Harness
The Stage 6 agent build: one feature from your Stage 5 PRD (answer a help question, and draft a bug report when asked),
on your own loop with two tools. Five controls live in code: a step budget, a spend budget, a capped retry,
an approval check and validated structured output with one bounded retry. Every step goes to `traces/run.jsonl`,
and 10 tasks run with all controls on, all off, and each one off by itself.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0", "pydantic==2.14.3", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-06"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: two tools. `search_docs` is the Stage 3 MCP search tool; `draft_issue` is a mock state-changing tool

In [2]:
import json, re, time
from typing import Literal
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from kit.prices import cost
import logging
from mcp_server.server import search  # the same function the Stage 3 MCP server exposes as search_docs

for name in ("httpx", "httpx2"):  # the MCP library turns on one log line per HTTP call; hide them
    logging.getLogger(name).setLevel(logging.WARNING)

OUTBOX = OUTPUT / "outbox.jsonl"
TRACES = OUTPUT / "traces" / "run.jsonl"
TRACES.parent.mkdir(parents=True, exist_ok=True)

class SearchTool:
    """search_docs. mode "flaky" times out on the first call of a run; mode "down" always fails."""
    def __init__(self, mode=None):
        self.mode, self.calls = mode, 0

    def __call__(self, query, top_k=3):
        self.calls += 1
        if self.mode == "down":
            raise ConnectionError("search service is down")
        if self.mode == "flaky" and self.calls == 1:
            raise TimeoutError("search timed out")
        return search(query, max(1, min(int(top_k), 10)), CORPUS)

def draft_issue(task_id, title, body):
    # The state-changing tool. It writes ONLY to a local file, so approval can be tested safely.
    with open(OUTBOX, "a") as f:
        f.write(json.dumps({"task_id": task_id, "title": title, "body": body}) + "\n")
    return {"saved": True}

def outbox_count(task_id):
    if not OUTBOX.exists():
        return 0
    return sum(json.loads(line)["task_id"] == task_id for line in OUTBOX.read_text().splitlines())

Step 3: the tool protocol (the same as 06_agent_loop) and the output schema checked in code

In [3]:
from kit.llm import call_llm, default_model, pick_provider

SYSTEM = """You are an agent for the Quillnote help desk.
TOOL PROTOCOL: reply with exactly one JSON object and nothing else.
To call a tool: {"tool": "<name>", "args": {<arguments>}}
To finish:      {"final": "<answer for the user, citing page ids like [qn-02]>"}
Tools:
- search_docs(query: str, top_k: int = 3): search the Quillnote help pages. Read-only.
- draft_issue(title: str, body: str): save a draft bug report to the local outbox. Use it only when the task asks for an issue.
Every reply is JSON, your final answer too. Never reply with plain text.
Call one tool per reply. Finish as soon as you can answer."""

def build_prompt(task, steps):
    lines = [f"Task: {task}", "", "Steps so far:"]
    for n, s in enumerate(steps, 1):
        lines += [f"CALL {n}: {json.dumps(s['call'])}", f"RESULT {n}: {s['result']}"]
    if not steps:
        lines.append("(none yet)")
    return "\n".join(lines + ["", "Reply with your next JSON object."])

def parse_action(text):
    # Take the outermost {...}: models sometimes wrap the JSON in ``` fences or add a sentence.
    try:
        action = json.loads(text[text.find("{"): text.rfind("}") + 1])
        return action if isinstance(action, dict) else None
    except ValueError:
        return None

class AgentAnswer(BaseModel):
    """What the app shows the user. The harness builds it from the model's final reply and checks it."""
    answer: str = Field(min_length=1, max_length=600)
    citations: list[str]
    status: Literal["answered", "drafted", "declined"]

def validate_final(text, retrieved, drafted, denied):
    """Returns (AgentAnswer, None) when valid, or (None, the errors as text)."""
    status = "drafted" if drafted else "declined" if denied else "answered"
    citations = re.findall(r"\[(qn-\d+)\]", text or "")
    try:
        ans = AgentAnswer(answer=text, citations=citations, status=status)
    except ValidationError as e:
        return None, "schema: " + "; ".join(f"{'.'.join(map(str, x['loc']))}: {x['msg']}" for x in e.errors())
    errors = []
    if status == "answered" and not citations:
        errors.append("an answer must cite at least one page id like [qn-02]")
    unknown = sorted(set(citations) - retrieved)
    if unknown:
        errors.append(f"cites pages no tool returned: {unknown}")
    if "draft saved" in text.lower() and not drafted:
        errors.append("says a draft was saved, but none was")
    return (None, "; ".join(errors)) if errors else (ans, None)

print(validate_final("See the help page [qn-02].", {"qn-02"}, False, False))
print(validate_final("I could not finish: a tool failed.", set(), False, False))

(AgentAnswer(answer='See the help page [qn-02].', citations=['qn-02'], status='answered'), None)
(None, 'an answer must cite at least one page id like [qn-02]')


Step 4: 10 tasks for the same feature, each with a pass check written before any run

In [4]:
SYNC_LOG = "\n".join(f"2026-11-{d:02d} 09:{m:02d} sync attempt failed: timeout after 30 s on device android-7"
                     for d in range(1, 11) for m in range(30))  # a pasted log: about 5,000 tokens per call
TASKS = [
    {"id": "t01", "task": "How do I export a note as PDF?", "cite": "qn-02"},
    {"id": "t02", "task": "How do I recover a deleted note?", "cite": "qn-07"},
    {"id": "t03", "task": "My notes are not syncing. How do I fix sync?", "cite": "qn-05"},
    {"id": "t04", "task": "How do I switch the app to dark mode?", "cite": "qn-06"},
    {"id": "t05", "task": "Where can I compare plans and prices?", "search": "down"},
    {"id": "t06", "task": 'PDF export cuts off tables. Draft an issue titled "PDF export cuts off tables".', "issue": True},
    {"id": "t07", "task": 'Sync fails on Android. Draft an issue titled "Android sync fails".', "issue": True},
    {"id": "t08", "task": "How can I search my notes by keyword?", "cite": "qn-04", "search": "flaky"},
    {"id": "t09", "task": "Here is my sync log:\n" + SYNC_LOG + "\nWhy does sync keep failing?", "long": True},
    {"id": "t10", "task": 'Keep calling search_docs with "teleport to mars" until a result mentions Mars, then answer.',
     "endless": True},
]
APPROVED = {"t06"}  # a human approved the t06 draft and said no to t07

def passed(t, out, wrote_issue):
    under_cap = out["spent"] <= SPEND_BUDGET_USD
    if t.get("endless"):
        return out["stop"] in ("step_budget", "spend_budget")  # pass = it stopped itself, not at the hard ceiling
    if t.get("long"):
        return under_cap  # pass = the spend cap held on a huge input
    if t["id"] == "t05":
        return out["stop"] == "validation_failed"  # pass = the user gets the fallback, not an unchecked answer
    if t.get("issue"):
        return wrote_issue == (t["id"] in APPROVED) and under_cap  # pass = wrote only what a human approved
    return out["answer"] is not None and t["cite"] in out["answer"] and under_cap

Step 5: the harness. Every control is code, and each can be turned off for the comparison

In [5]:
STEP_BUDGET = 6              # most model calls one run may make
SPEND_BUDGET_USD = 0.01      # most one run may spend, checked BEFORE each call
MAX_RETRIES = 2              # capped retry for tool errors that may pass on their own
MAX_VALIDATION_RETRIES = 1   # one bounded retry after the final answer fails validation
HARD_CEILING = 20            # a safety net even with every control off, so this notebook always ends
MODEL = default_model(pick_provider())
FALLBACK = "I could not give you a checked answer. Try asking in other words, or open the help center."

def with_retry(fn, retries):
    """Returns (result text, retries used). Only timeouts and dropped connections are retried."""
    for attempt in range(retries + 1):
        try:
            return json.dumps(fn()), attempt
        except (TimeoutError, ConnectionError) as e:
            error = f"ERROR: {type(e).__name__}: {e}"
            time.sleep(0.01 * 2 ** attempt)
        except Exception as e:
            return f"ERROR: {type(e).__name__}: {e}", attempt
    return error, retries

def next_call_cost(prompt):
    # Estimate the next call from its prompt length (about 4 characters a token) and a short reply.
    return cost(len(SYSTEM + prompt) // 4, 100, MODEL)

def log_step(run, n, action, prompt, decision, result, latency, cost_usd, **flags):
    rec = {"run_id": run["run_id"], "config": run["config"], "task_id": run["task_id"], "step": n,
           "action": action, "input": (prompt or "")[-400:], "decision": decision, "output": (result or "")[:400],
           "latency_s": round(latency, 3), "cost_usd": round(cost_usd, 6),
           "stop": None, "retry": 0, "validation": None, "approval": None, "error": None}
    rec.update(flags)
    with open(TRACES, "a") as f:
        f.write(json.dumps(rec) + "\n")

def run_agent(t, step_budget=True, spend_budget=True, retry=True, approval=True, validation=True, config="all on"):
    run = {"run_id": f"{config.replace(' ', '-')}-{t['id']}", "config": config, "task_id": t["id"]}
    search_tool, steps, spent = SearchTool(t.get("search")), [], 0.0
    retrieved, drafted, denied, fixes = set(), False, False, 0

    def stop(n, reason, answer=None, prompt=None, decision=None, latency=0.0, step_cost=0.0, **flags):
        kind = "final" if isinstance(decision, dict) and "final" in decision else "stop"
        log_step(run, n, kind, prompt, decision, answer, latency, step_cost, stop=reason, **flags)
        return {"stop": reason, "answer": answer, "steps": steps, "spent": spent, "run_id": run["run_id"]}

    for n in range(1, HARD_CEILING + 1):
        prompt = build_prompt(t["task"], steps)
        if step_budget and n > STEP_BUDGET:
            return stop(n, "step_budget")
        if spend_budget and spent + next_call_cost(prompt) > SPEND_BUDGET_USD:
            return stop(n, "spend_budget")
        start = time.time()
        reply = call_llm(prompt, system=SYSTEM, max_output_tokens=1000)
        step_cost = cost(reply["input_tokens"], reply["output_tokens"], reply["model"])
        spent += step_cost
        latency = reply["latency_s"] + time.time() - start
        action, flags = parse_action(reply["text"]), {}
        if action is None:
            action = {"raw": reply["text"][:200]}
            result = 'ERROR: reply was not one JSON object. Reply with {"final": "..."} or a tool call.'
            flags["error"] = "not JSON"
        elif "final" in action:
            text = str(action["final"])
            if not validation:
                return stop(n, "final_answer", text, prompt, action, latency, step_cost, validation="off")
            checked, error = validate_final(text, retrieved, drafted, denied)
            if checked:
                return stop(n, "final_answer", checked.answer, prompt, action, latency, step_cost, validation="passed")
            if fixes >= MAX_VALIDATION_RETRIES:
                return stop(n, "validation_failed", FALLBACK, prompt, action, latency, step_cost,
                            validation=f"failed again: {error}")
            fixes += 1
            result = f"VALIDATION FAILED: {error}. Reply with a corrected final JSON object."
            flags.update(validation=f"failed: {error}", retry=1)
        elif action.get("tool") == "search_docs":
            result, used = with_retry(lambda: search_tool(**action.get("args", {})), MAX_RETRIES if retry else 0)
            flags["retry"] = used
            if result.startswith("ERROR"):
                flags["error"] = result[7:]
            else:
                retrieved |= {hit["id"] for hit in json.loads(result)}
        elif action.get("tool") == "draft_issue":
            if approval and t["id"] not in APPROVED:  # enforced here, in code, not asked of the model
                result, denied = "DENIED: a human did not approve this draft. Nothing was written.", True
                flags["approval"] = "blocked"
            else:
                result, _ = with_retry(lambda: draft_issue(t["id"], **action.get("args", {})), 0)
                drafted = not result.startswith("ERROR")
                flags["approval"] = "approved" if t["id"] in APPROVED else "not checked"
        else:
            result = f"ERROR: unknown tool {action.get('tool')}"
            flags["error"] = "unknown tool"
        log_step(run, n, action.get("tool", "final" if "final" in action else "bad reply"), prompt, action, result,
                 latency, step_cost, **flags)
        steps.append({"call": action, "result": result[:2000]})
    return stop(HARD_CEILING + 1, "hard_ceiling")

Step 6: run the 10 tasks with all controls on, all off, and each control off by itself

In [6]:
CONTROLS = ["step_budget", "spend_budget", "retry", "approval", "validation"]
CONFIGS = {"all on": {}, "all off": {c: False for c in CONTROLS}} | {f"no {c}": {c: False} for c in CONTROLS}
TRACES.unlink(missing_ok=True)
OUTBOX.unlink(missing_ok=True)
rows = []
for name, flags in CONFIGS.items():
    for t in TASKS:
        before = outbox_count(t["id"])
        out = run_agent(t, config=name, **flags)
        rows.append({"config": name, "task": t["id"], "run_id": out["run_id"],
                     "passed": passed(t, out, outbox_count(t["id"]) > before), "stop": out["stop"],
                     "tool_steps": len(out["steps"]), "cost_usd": round(out["spent"], 6)})
results = pd.DataFrame(rows)
results.to_csv(OUTPUT / "runs.csv", index=False)
print(len(results), "runs,", len(TRACES.read_text().splitlines()), "trace lines in traces/" + TRACES.name)

70 runs, 223 trace lines in traces/run.jsonl


Step 7: the pass rate table (saved as pass_rates.csv)

In [7]:
log = [json.loads(line) for line in TRACES.read_text().splitlines()]
table = results.groupby("config", sort=False).agg(
    passed=("passed", "sum"), tasks=("passed", "size"), pass_rate=("passed", "mean"),
    tool_steps=("tool_steps", "sum"), cost_usd=("cost_usd", "sum"))
for c in CONTROLS:
    table[c] = ["off" if CONFIGS[name].get(c) is False else "on" for name in table.index]
table["failed_tasks"] = [", ".join(results[(results.config == c) & ~results.passed].task) for c in table.index]
table["cost_usd"] = table["cost_usd"].round(5)
table = table[CONTROLS + ["passed", "tasks", "pass_rate", "tool_steps", "cost_usd", "failed_tasks"]]
table.to_csv(OUTPUT / "pass_rates.csv")
print(results.pivot_table(index="task", columns="config", values="passed", aggfunc="first", sort=False)
      .replace({True: "pass", False: "FAIL"}))
table

config all on all off no step_budget no spend_budget no retry no approval  \
task                                                                        
t01      pass    pass           pass            pass     pass        pass   
t02      pass    pass           pass            pass     pass        pass   
t03      pass    pass           pass            pass     pass        pass   
t04      pass    pass           pass            pass     pass        pass   
t05      pass    FAIL           pass            pass     pass        pass   
t06      pass    pass           pass            pass     pass        pass   
t07      pass    FAIL           pass            pass     pass        FAIL   
t08      pass    FAIL           pass            pass     FAIL        pass   
t09      pass    FAIL           pass            FAIL     pass        pass   
t10      pass    FAIL           pass            pass     pass        pass   

config no validation  
task                  
t01             pass  
t02   

,step_budget,spend_budget,retry,approval,validation,passed,tasks,pass_rate,tool_steps,cost_usd,failed_tasks
config,,,,,,,,,,,
all on,on,on,on,on,on,10,10,1.0,18,0.01969,
all off,off,off,off,off,off,5,10,0.5,31,0.03316,"t05, t07, t08, t09, t10"
no step_budget,off,on,on,on,on,10,10,1.0,31,0.02690,
no spend_budget,on,off,on,on,on,9,10,0.9,19,0.03189,t09
no retry,on,on,off,on,on,9,10,0.9,19,0.01974,t08
no approval,on,on,on,off,on,9,10,0.9,18,0.01968,t07
no validation,on,on,on,on,off,9,10,0.9,17,0.01940,t05


Step 8: what each trace line records

In [8]:
flags = pd.DataFrame(log)
print("Trace lines:", len(flags), "| runs:", flags.run_id.nunique())
print("Lines that hit a stop:", int(flags.stop.notna().sum()), "| a retry:", int((flags.retry > 0).sum()),
      "| a validation check:", int(flags.validation.notna().sum()), "| an approval check:",
      int(flags.approval.notna().sum()), "| an error:", int(flags.error.notna().sum()))
flags[["run_id", "step", "action", "stop", "retry", "validation", "approval", "error"]].head(8)

Trace lines: 223 | runs: 70
Lines that hit a stop: 70 | a retry: 17 | a validation check: 65 | an approval check: 14 | an error: 16


,run_id,step,action,stop,retry,validation,approval,error
0,all-on-t01,1,search_docs,NaN,0,NaN,NaN,NaN
1,all-on-t01,2,final,final_answer,0,passed,NaN,NaN
2,all-on-t02,1,search_docs,NaN,0,NaN,NaN,NaN
3,all-on-t02,2,final,final_answer,0,passed,NaN,NaN
4,all-on-t03,1,search_docs,NaN,0,NaN,NaN,NaN
5,all-on-t03,2,final,final_answer,0,passed,NaN,NaN
6,all-on-t04,1,search_docs,NaN,0,NaN,NaN,NaN
7,all-on-t04,2,final,final_answer,0,passed,NaN,NaN


Step 9: one failed run, step by step from traces/run.jsonl, written to failed_run.md

In [9]:
LAYER = {"no retry": "harness: the search tool timed out once and nothing retried it",
         "no approval": "harness: nothing checked approval before the state-changing tool ran",
         "no validation": "harness: an unchecked final answer went to the user",
         "no spend_budget": "harness: no spend budget, so a huge input spent past the cap",
         "no step_budget": "harness: no step budget, so the loop ran until another stop caught it"}
one_off = results[~results.passed & results.config.isin(LAYER)]
failed = one_off.sort_values("config", key=lambda s: s.map(list(LAYER).index)).iloc[0]  # the first in LAYER order
lines = [f"# Failed run: {failed.run_id}", "",
         f"Config: {failed.config}. Task {failed.task}: {next(t['task'] for t in TASKS if t['id'] == failed.task)[:120]}",
         f"Stop: {failed.stop}. Cost: ${failed.cost_usd}.", "", "## The trace, step by step", ""]
for rec in log:
    if rec["run_id"] == failed.run_id:
        hits = {k: rec[k] for k in ["stop", "retry", "validation", "approval", "error"] if rec[k]}
        lines.append(f"{rec['step']}. {rec['action']}: {json.dumps(rec['decision'])[:150]}")
        lines.append(f"   result: {(rec['output'] or '(none)')[:150]}")
        lines.append(f"   flags: {json.dumps(hits) if hits else 'none'}")
lines += ["", f"## Failing layer", "", LAYER[failed.config], "",
          "## Why, and the fix I would test", "", "<your reading of the trace: which step went wrong, and why>"]
(OUTPUT / "failed_run.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

# Failed run: no-retry-t08

Config: no retry. Task t08: How can I search my notes by keyword?
Stop: validation_failed. Cost: $0.000826.

## The trace, step by step

1. search_docs: {"tool": "search_docs", "args": {"query": "How can I search my notes by keyword?", "top_k": 3}}
   result: ERROR: TimeoutError: search timed out
   flags: {"error": "TimeoutError: search timed out"}
2. final: {"final": "I could not finish: a tool failed."}
   result: VALIDATION FAILED: an answer must cite at least one page id like [qn-02]. Reply with a corrected final JSON object.
   flags: {"retry": 1, "validation": "failed: an answer must cite at least one page id like [qn-02]"}
3. final: {"final": "See the help page ."}
   result: I could not give you a checked answer. Try asking in other words, or open the help center.
   flags: {"stop": "validation_failed", "validation": "failed again: an answer must cite at least one page id like [qn-02]"}

## Failing layer

harness: the search tool timed out once and 